# Resolver report

Data comes from `reports/resolver.duckdb`, refreshed by `scripts/report.sh` before this notebook opens.

Run all cells (**Kernel → Restart & Run All**) to see the current state of your benchmark runs.

In [ ]:
from pathlib import Path
import os
import duckdb


def _repo_root():
    here = Path(os.getcwd()).resolve()
    for p in [here, *here.parents]:
        if (p / "go.mod").exists():
            return p
    raise RuntimeError("repo root (go.mod) not found — run via scripts/report.sh")


ROOT = _repo_root()
DB = ROOT / "reports" / "resolver.duckdb"
if not DB.exists():
    raise RuntimeError(f"{DB} missing — run scripts/report.sh first")

con = duckdb.connect(str(DB), read_only=True)
print(f"connected: {DB}")

## What's in the database?

In [ ]:
con.sql("SHOW TABLES").to_df()

## Runs — one row per benchmark run

`virtual_model` is the harness-facing name (what you pass as `--model`); `real_model` is the backing model from the sidecar `run-config.yaml`. `p95_ms` is the 95th-percentile per-query latency. v2.1+ no longer surfaces a monolithic `overall` verdict — see the role-coverage heat-map below for PASS/FAIL per role.

In [ ]:
con.sql("""
    SELECT run_id, model AS virtual_model, cfg_real_model AS real_model,
           correct_count, query_count, p95_ms
    FROM run_summary
    ORDER BY run_id DESC
""").to_df()

## Role-coverage heat-map

Rows are a **MultiIndex on `(real_model, virtual_model)`** — virtuals that share a backing real model cluster under the same top-level group (e.g. gresh-general + gresh-coder + gresh-reasoner all sit under `Qwen/Qwen3.6-35B-A3B-FP8` since they're different proxy-route tunings of the same weights).

Columns: one of the v2.1 roles. Cells: most recent verdict (PASS green, FAIL red, ERROR amber, INFO blue, missing transparent). The real_model column is sourced from `run_config.real_model` (the sidecar); `resolved_real_model` (from the `/v1/models` probe) is blank on private endpoints due to the SSRF guard and is therefore not used here.

In [ ]:
import pandas as pd

df = con.sql("""
    SELECT
        COALESCE(NULLIF(rc_cfg.real_model, ''), rc.model) AS real_model,
        rc.model   AS virtual_model,
        rcv.role,
        rcv.verdict,
        rcv.run_id
    FROM role_coverage rcv
    JOIN runs rc ON rc.run_id = rcv.run_id
    LEFT JOIN run_config rc_cfg ON rc_cfg.run_id = rcv.run_id
    ORDER BY rcv.run_id DESC
""").to_df()

recent = df.drop_duplicates(subset=["real_model", "virtual_model", "role"], keep="first")

pivot = recent.pivot_table(
    index=["real_model", "virtual_model"],
    columns="role",
    values="verdict",
    aggfunc="first",
).sort_index(level=[0, 1])


def _color(v):
    if v == "PASS":  return "background-color: #4caf50; color: white"
    if v == "FAIL":  return "background-color: #e53935; color: white"
    if v == "ERROR": return "background-color: #fb8c00; color: white"
    if v == "INFO":  return "background-color: #1e88e5; color: white"
    return "background-color: transparent"


pivot.style.map(_color).set_caption(
    "Role-coverage heat-map — rows grouped by backing real_model; virtuals below"
)

### Same shape, observed % per cell

Read-through for the heat-map. Cells are the `pct` key from `role_scorecards.metrics_json`, `-1` when a role produced no scored scenario (long-context hard-fail, reducer-sexp inline-skip, etc.).

In [ ]:
import json as _json

def _pct(s):
    try: return int(_json.loads(s or "{}").get("pct", -1))
    except Exception: return -1

df_pct = con.sql("""
    SELECT
        COALESCE(NULLIF(rc_cfg.real_model, ''), rs.model) AS real_model,
        rs.model        AS virtual_model,
        rscore.role,
        rscore.metrics_json,
        rscore.threshold,
        rs.run_id
    FROM role_scorecards rscore
    JOIN runs rs ON rs.run_id = rscore.run_id
    LEFT JOIN run_config rc_cfg ON rc_cfg.run_id = rscore.run_id
    ORDER BY rs.run_id DESC
""").to_df()

df_pct["pct"] = df_pct["metrics_json"].map(_pct)
recent_pct = df_pct.drop_duplicates(
    subset=["real_model", "virtual_model", "role"], keep="first"
)
recent_pct.pivot_table(
    index=["real_model", "virtual_model"],
    columns="role",
    values="pct",
    aggfunc="first",
).sort_index(level=[0, 1]).fillna(-1).astype(int)

### Per-role thresholds + gate decision

`threshold_met` is the authoritative gate decision. `observed` vs `expected` scenario counts surface mid-run drops (HF rate limits, hard-fail skips).

In [ ]:
con.sql("""
    SELECT run_id, role, verdict, threshold_met, threshold,
           scenario_count_expected AS expected,
           scenario_count_observed AS observed
    FROM role_coverage
    ORDER BY run_id DESC, role
""").to_df()

## Per-query results

Flat view: one row per (run, scenario). Limited to the most recent 50 rows; edit the SQL to widen.

In [ ]:
con.sql("""
    SELECT run_id, model AS virtual_model,
           cfg_real_model AS real_model,
           scenario_id, score, elapsed_ms
    FROM comparison
    ORDER BY run_id DESC, scenario_id
    LIMIT 50
""").to_df()

## Community benchmarks

Reference scores from the broader LLM ecosystem. `model_key` is the normalised form used to join against the real model name.

In [ ]:
con.sql("""
    SELECT model, model_key, benchmark, metric, value, as_of
    FROM community_benchmarks
    ORDER BY model, benchmark
""").to_df()

### Model × benchmark matrix

Pivoted view — rows are normalised `model_key`, columns are `benchmark/metric` pairs.

In [ ]:
df = con.sql("""
    SELECT model_key,
           benchmark || '/' || metric AS bench_metric,
           value
    FROM community_benchmarks
""").to_df()

df.pivot_table(index="model_key", columns="bench_metric", values="value", aggfunc="first")

## Next

- Drill into a single run in **`reproducibility.ipynb`**.
- Write your own SQL below — `con` is a read-only DuckDB connection and `.to_df()` renders a pandas DataFrame.
- Re-run `scripts/report.sh` after new benchmark runs to refresh.